# Clustering V1 — Apartment Complex Segmentation

## 목적

서울 아파트 단지를 가격 자체가 아닌
**주택 구조 + 입지 특성**을 기준으로 유형화한다.

군집화의 목적은 예측이 아니라 다음과 같다.

- 비슷한 특성을 가진 아파트 단지 탐색
- 단지 유형별 특성 비교
- Regression 결과의 해석 보조
- 웹 서비스에서 단지 유형 설명

## 기본 원칙

- 단위: 거래가 아닌 **아파트 단지**
- 학습기간: 2021-07 ~ 2025-12
- Test 기간: 2026-01 ~ 2026-06
- 가격(`거래금액`, `평당가`)은 군집 Feature에서 제외
- 가격은 최종 군집을 만든 후 사후 해석에만 사용
- 거리 Feature는 log1p 변환 검토
- Feature Scaling 적용
- KMeans / GMM / Agglomerative 비교

## 평가 기준

1. Silhouette Score — 높을수록 좋음
2. Davies-Bouldin Index — 낮을수록 좋음
3. Calinski-Harabasz Score — 높을수록 좋음
4. 군집 크기 균형
5. 군집별 Silhouette
6. 비즈니스 해석 가능성

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
)


plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


PROJECT_DIR = Path.cwd().parents[3]

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
    / "df_final_v1.parquet"
)

df = pd.read_parquet(DATA_PATH)

df["거래월일자"] = pd.to_datetime(
    df["거래월일자"]
)

print("전체 데이터:", df.shape)
print(
    "기간:",
    df["거래월일자"].min(),
    "~",
    df["거래월일자"].max(),
)

전체 데이터: (234302, 124)
기간: 2021-07-01 00:00:00 ~ 2026-06-01 00:00:00


In [3]:
# ============================================================
# Clustering Train / Test
# ============================================================

cluster_train_df = df[
    df["거래월일자"] < "2026-01-01"
].copy()

cluster_test_df = df[
    df["거래월일자"] >= "2026-01-01"
].copy()


print(
    "Train 거래:",
    f"{len(cluster_train_df):,}",
)

print(
    "Test 거래:",
    f"{len(cluster_test_df):,}",
)

print(
    "Train 단지:",
    f"{cluster_train_df['단지일련번호'].nunique():,}",
)

print(
    "Test 단지:",
    f"{cluster_test_df['단지일련번호'].nunique():,}",
)

Train 거래: 194,733
Test 거래: 39,569
Train 단지: 7,581
Test 단지: 5,038


## 시간 분리 기준

군집화는 비지도학습이지만,
2026년 데이터를 이용해 미리 단지 유형을 결정하지 않도록
2025년 12월까지의 데이터만 사용해 군집 구조를 학습한다.

2026년 상반기 데이터는 다음 용도로만 사용한다.

- 기존 단지 군집 연결 Coverage 확인
- 신규 단지 일반화 가능성 확인
- Regression V1의 군집별 성능 분석

이를 통해 Clustering도 Regression / Classification과 동일하게
시간 기준을 명확하게 유지한다.

In [4]:
# ============================================================
# 거래 단위 → 단지 단위
# ============================================================

complex_df = (
    cluster_train_df
    .groupby("단지일련번호")
    .agg(
        단지명=("단지명", "first"),
        법정동=("법정동", "first"),

        거래건수=("transaction_id", "size"),

        전용면적=("전용면적", "median"),
        건축년도=("건축년도", "median"),
        층=("층", "median"),

        유치원거리_m=("유치원거리_m", "median"),
        초등학교거리_m=("초등학교거리_m", "median"),
        중학교거리_m=("중학교거리_m", "median"),
        고등학교거리_m=("고등학교거리_m", "median"),

        역세권거리_m=("역세권거리_m", "median"),

        응급의료기관거리_m=(
            "응급의료기관거리_m",
            "median",
        ),

        대규모점포거리_m=(
            "대규모점포거리_m",
            "median",
        ),

        주요일자리거리_m=(
            "주요일자리거리_m",
            "median",
        ),
    )
    .reset_index()
)


# 2025년 말 기준 연식
complex_df["기준연식"] = (
    2025
    - complex_df["건축년도"]
).clip(lower=0)


# 교육시설 종합 접근거리
school_cols = [
    "유치원거리_m",
    "초등학교거리_m",
    "중학교거리_m",
    "고등학교거리_m",
]

complex_df["교육접근거리_m"] = (
    complex_df[school_cols]
    .mean(axis=1)
)


print(
    "군집 대상 단지:",
    f"{len(complex_df):,}",
)

display(
    complex_df.head()
)

군집 대상 단지: 7,581


,단지일련번호,단지명,법정동,거래건수,전용면적,건축년도,층,유치원거리_m,초등학교거리_m,중학교거리_m,고등학교거리_m,역세권거리_m,응급의료기관거리_m,대규모점포거리_m,주요일자리거리_m,기준연식,교육접근거리_m
0,11110-101,크래스빌,평창동,2,185.20,2003.0,3.5,712.566751,1296.503413,1747.844193,635.342932,3020.956750,3720.835999,3438.306788,3954.796658,22.0,1098.064322
1,11110-102,현대뜨레비앙,익선동,56,46.64,2003.0,5.0,238.146726,168.497780,776.048260,644.051185,287.541968,864.937610,385.989878,19.536114,22.0,456.685988
2,11110-107,경희궁파크팰리스,내수동,13,146.33,2003.0,13.0,145.597501,487.673416,662.693783,662.693783,325.663252,671.265175,774.784827,1153.013723,22.0,489.664621
3,11110-111,창림,창신동,1,58.62,1994.0,3.0,375.649908,434.520432,480.484225,480.583185,498.109449,1002.851597,590.648630,1721.153827,31.0,442.809437
4,11110-112,신현(102동),신교동,1,121.96,2002.0,5.0,234.843652,293.323506,438.529646,407.186761,969.192395,1287.057291,1674.854056,2120.624419,23.0,343.470891


In [6]:
complex_df["유초등접근거리_m"] = (
    complex_df[
        [
            "유치원거리_m",
            "초등학교거리_m",
        ]
    ]
    .mean(axis=1)
)

In [7]:
cluster_feature_sets = {

    "A_주택구조": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "층",
    ],

    "B_교육생활": [
        "유초등접근거리_m",
        "중학교거리_m",
        "고등학교거리_m",
        "대규모점포거리_m",
        "응급의료기관거리_m",
    ],

    "C_교통업무": [
        "역세권거리_m",
        "주요일자리거리_m",
    ],

    "D_핵심통합": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "유초등접근거리_m",
        "역세권거리_m",
        "주요일자리거리_m",
    ],

    "E_전체통합": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "층",

        "유초등접근거리_m",
        "중학교거리_m",
        "고등학교거리_m",

        "역세권거리_m",
        "대규모점포거리_m",
        "응급의료기관거리_m",
        "주요일자리거리_m",
    ],
}

In [8]:
# ============================================================
# 거래 단위 → 단지 단위
# ============================================================

complex_df = (
    cluster_train_df
    .groupby("단지일련번호")
    .agg(
        단지명=("단지명", "first"),
        법정동=("법정동", "first"),

        거래건수=("transaction_id", "size"),

        전용면적=("전용면적", "median"),
        전용면적_Q25=("전용면적", lambda x: x.quantile(0.25)),
        전용면적_Q75=("전용면적", lambda x: x.quantile(0.75)),

        건축년도=("건축년도", "median"),
        층=("층", "median"),

        유치원거리_m=("유치원거리_m", "median"),
        초등학교거리_m=("초등학교거리_m", "median"),
        중학교거리_m=("중학교거리_m", "median"),
        고등학교거리_m=("고등학교거리_m", "median"),

        역세권거리_m=("역세권거리_m", "median"),
        응급의료기관거리_m=("응급의료기관거리_m", "median"),
        대규모점포거리_m=("대규모점포거리_m", "median"),
        주요일자리거리_m=("주요일자리거리_m", "median"),
    )
    .reset_index()
)


# 2025년 말 기준 연식
complex_df["기준연식"] = (
    2025 - complex_df["건축년도"]
).clip(lower=0)


# 단지 내 면적 구성 다양성
complex_df["전용면적_IQR"] = (
    complex_df["전용면적_Q75"]
    - complex_df["전용면적_Q25"]
)


# 유치원 + 초등학교 접근성
complex_df["유초등접근거리_m"] = (
    complex_df[
        [
            "유치원거리_m",
            "초등학교거리_m",
        ]
    ]
    .mean(axis=1)
)


print("군집 대상 단지:", f"{len(complex_df):,}")

군집 대상 단지: 7,581


In [13]:
cluster_feature_sets = {

    "A_주택구조": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "층",
    ],

    "B_교육생활": [
        "유초등접근거리_m",
        "대규모점포거리_m",
        "응급의료기관거리_m",
    ],

    "C_교통업무": [
        "역세권거리_m",
        "주요일자리거리_m",
    ],

    "D_핵심통합": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "유초등접근거리_m",
        "역세권거리_m",
        "주요일자리거리_m",
    ],

    "E_전체통합": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "층",

        "유초등접근거리_m",
        "역세권거리_m",

        "대규모점포거리_m",
        "응급의료기관거리_m",
        "주요일자리거리_m",
    ],
}

In [14]:
# ============================================================
# Clustering 전처리
# ============================================================

LOG_COLS = {
    "전용면적",
    "전용면적_IQR",
    "유초등접근거리_m",
    "역세권거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
    "주요일자리거리_m",
}

def prepare_cluster_data(data, feature_cols):

    model_df = data[feature_cols].copy()

    # 음수 방지
    for col in feature_cols:
        if col in LOG_COLS:
            model_df[col] = np.log1p(
                model_df[col].clip(lower=0)
            )

    scaler = StandardScaler()

    X = scaler.fit_transform(
        model_df
    )

    return X, scaler

In [15]:
# ============================================================
# KMeans / GMM / Agglomerative 비교
# ============================================================

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)


comparison_results = []


for feature_set_name, feature_cols in cluster_feature_sets.items():

    X, _ = prepare_cluster_data(
        complex_df,
        feature_cols,
    )


    for k in range(2, 7):

        # ----------------------------------------------------
        # KMeans
        # ----------------------------------------------------

        kmeans = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=42,
        )

        kmeans_labels = kmeans.fit_predict(X)


        comparison_results.append({
            "Feature_Set": feature_set_name,
            "Algorithm": "KMeans",
            "K": k,

            "Silhouette": silhouette_score(
                X,
                kmeans_labels,
                sample_size=min(3000, len(X)),
                random_state=42,
            ),

            "Davies_Bouldin": davies_bouldin_score(
                X,
                kmeans_labels,
            ),

            "Calinski_Harabasz": calinski_harabasz_score(
                X,
                kmeans_labels,
            ),
        })


        # ----------------------------------------------------
        # GMM
        # ----------------------------------------------------

        gmm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=42,
            n_init=3,
        )

        gmm_labels = gmm.fit_predict(X)


        comparison_results.append({
            "Feature_Set": feature_set_name,
            "Algorithm": "GMM",
            "K": k,

            "Silhouette": silhouette_score(
                X,
                gmm_labels,
                sample_size=min(3000, len(X)),
                random_state=42,
            ),

            "Davies_Bouldin": davies_bouldin_score(
                X,
                gmm_labels,
            ),

            "Calinski_Harabasz": calinski_harabasz_score(
                X,
                gmm_labels,
            ),
        })


        # ----------------------------------------------------
        # Agglomerative
        # ----------------------------------------------------

        agg = AgglomerativeClustering(
            n_clusters=k,
            linkage="ward",
        )

        agg_labels = agg.fit_predict(X)


        comparison_results.append({
            "Feature_Set": feature_set_name,
            "Algorithm": "Agglomerative",
            "K": k,

            "Silhouette": silhouette_score(
                X,
                agg_labels,
                sample_size=min(3000, len(X)),
                random_state=42,
            ),

            "Davies_Bouldin": davies_bouldin_score(
                X,
                agg_labels,
            ),

            "Calinski_Harabasz": calinski_harabasz_score(
                X,
                agg_labels,
            ),
        })


cluster_comparison_df = pd.DataFrame(
    comparison_results
)


display(
    cluster_comparison_df
    .sort_values(
        "Silhouette",
        ascending=False,
    )
    .head(30)
    .round(4)
)

,Feature_Set,Algorithm,K,Silhouette,Davies_Bouldin,Calinski_Harabasz
16,B_교육생활,GMM,2,0.5010,1.4241,1206.6895
31,C_교통업무,GMM,2,0.3860,1.1802,4073.2876
30,C_교통업무,KMeans,2,0.3774,1.1384,4394.6136
33,C_교통업무,KMeans,3,0.3671,0.9032,5158.0597
42,C_교통업무,KMeans,6,0.3426,0.8578,4947.5662
39,C_교통업무,KMeans,5,0.3370,0.8909,4969.2898
35,C_교통업무,Agglomerative,3,0.3349,1.0024,4273.9584
3,A_주택구조,KMeans,3,0.3320,1.1818,2786.1853
9,A_주택구조,KMeans,5,0.3305,1.1462,2469.6637
43,C_교통업무,GMM,6,0.3279,0.8944,4456.0920


In [16]:
# ============================================================
# 각 알고리즘 / Feature Set별 최고 조합
# ============================================================

best_cluster_candidates = (
    cluster_comparison_df
    .sort_values(
        "Silhouette",
        ascending=False,
    )
    .groupby(
        [
            "Feature_Set",
            "Algorithm",
        ],
        as_index=False,
    )
    .first()
    .sort_values(
        "Silhouette",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    best_cluster_candidates.round(4)
)

,Feature_Set,Algorithm,K,Silhouette,Davies_Bouldin,Calinski_Harabasz
0,B_교육생활,GMM,2,0.5010,1.4241,1206.6895
1,C_교통업무,GMM,2,0.3860,1.1802,4073.2876
2,C_교통업무,KMeans,2,0.3774,1.1384,4394.6136
3,C_교통업무,Agglomerative,3,0.3349,1.0024,4273.9584
4,A_주택구조,KMeans,3,0.3320,1.1818,2786.1853
5,E_전체통합,Agglomerative,2,0.2951,1.8441,646.0881
6,A_주택구조,Agglomerative,6,0.2873,1.2681,1936.9557
7,B_교육생활,KMeans,4,0.2719,0.9905,2958.0576
8,B_교육생활,Agglomerative,3,0.2272,1.2051,2011.2716
9,D_핵심통합,KMeans,4,0.1968,1.6122,1351.7109


In [17]:
final_candidates = [
    {
        "name": "B_교육생활_GMM_K2",
        "feature_set": "B_교육생활",
        "algorithm": "GMM",
        "k": 2,
    },
    {
        "name": "C_교통업무_GMM_K2",
        "feature_set": "C_교통업무",
        "algorithm": "GMM",
        "k": 2,
    },
    {
        "name": "C_교통업무_KMeans_K2",
        "feature_set": "C_교통업무",
        "algorithm": "KMeans",
        "k": 2,
    },
    {
        "name": "A_주택구조_KMeans_K3",
        "feature_set": "A_주택구조",
        "algorithm": "KMeans",
        "k": 3,
    },
]


candidate_labels = {}
candidate_models = {}
candidate_scalers = {}


for candidate in final_candidates:

    feature_cols = cluster_feature_sets[
        candidate["feature_set"]
    ]

    X, scaler = prepare_cluster_data(
        complex_df,
        feature_cols,
    )

    if candidate["algorithm"] == "KMeans":

        model = KMeans(
            n_clusters=candidate["k"],
            n_init=20,
            random_state=42,
        )

        labels = model.fit_predict(X)

    else:

        model = GaussianMixture(
            n_components=candidate["k"],
            covariance_type="full",
            n_init=3,
            random_state=42,
        )

        labels = model.fit_predict(X)


    candidate_labels[candidate["name"]] = labels
    candidate_models[candidate["name"]] = model
    candidate_scalers[candidate["name"]] = scaler

    counts = (
        pd.Series(labels)
        .value_counts()
        .sort_index()
    )

    print(f"\n===== {candidate['name']} =====")

    for cluster, count in counts.items():

        print(
            f"Cluster {cluster}: "
            f"{count:,}개 "
            f"({count / len(labels) * 100:.2f}%)"
        )


===== B_교육생활_GMM_K2 =====
Cluster 0: 7,165개 (94.51%)
Cluster 1: 416개 (5.49%)

===== C_교통업무_GMM_K2 =====
Cluster 0: 5,139개 (67.79%)
Cluster 1: 2,442개 (32.21%)

===== C_교통업무_KMeans_K2 =====
Cluster 0: 4,928개 (65.00%)
Cluster 1: 2,653개 (35.00%)

===== A_주택구조_KMeans_K3 =====
Cluster 0: 2,757개 (36.37%)
Cluster 1: 3,783개 (49.90%)
Cluster 2: 1,041개 (13.73%)


In [18]:
candidate_silhouette_results = []


for candidate in final_candidates:

    name = candidate["name"]

    feature_cols = cluster_feature_sets[
        candidate["feature_set"]
    ]

    X, _ = prepare_cluster_data(
        complex_df,
        feature_cols,
    )

    labels = candidate_labels[name]

    silhouette_values = silhouette_samples(
        X,
        labels,
    )


    for cluster in np.unique(labels):

        mask = labels == cluster

        candidate_silhouette_results.append({
            "후보": name,
            "Cluster": cluster,
            "단지수": int(mask.sum()),
            "단지비율_%": mask.mean() * 100,
            "평균_Silhouette": silhouette_values[mask].mean(),
            "음수비율_%": (
                silhouette_values[mask] < 0
            ).mean() * 100,
        })


candidate_silhouette_df = pd.DataFrame(
    candidate_silhouette_results
)

display(
    candidate_silhouette_df.round(3)
)

,후보,Cluster,단지수,단지비율_%,평균_Silhouette,음수비율_%
0,B_교육생활_GMM_K2,0,7165,94.513,0.512,0.028
1,B_교육생활_GMM_K2,1,416,5.487,0.054,34.856
2,C_교통업무_GMM_K2,0,5139,67.788,0.481,0.000
3,C_교통업무_GMM_K2,1,2442,32.212,0.191,17.813
4,C_교통업무_KMeans_K2,0,4928,65.005,0.467,0.000
5,C_교통업무_KMeans_K2,1,2653,34.995,0.217,12.778
6,A_주택구조_KMeans_K3,0,2757,36.367,0.277,2.793
7,A_주택구조_KMeans_K3,1,3783,49.901,0.400,0.000
8,A_주택구조_KMeans_K3,2,1041,13.732,0.229,14.697


In [19]:
profile_cols = [
    "전용면적",
    "전용면적_IQR",
    "기준연식",
    "층",
    "유초등접근거리_m",
    "역세권거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
    "주요일자리거리_m",
]


for candidate in final_candidates:

    name = candidate["name"]

    temp = complex_df.copy()
    temp["cluster"] = candidate_labels[name]

    profile = (
        temp
        .groupby("cluster")[profile_cols]
        .median()
    )

    profile["단지수"] = (
        temp
        .groupby("cluster")
        .size()
    )

    profile["단지비율_%"] = (
        profile["단지수"]
        / len(temp)
        * 100
    )

    print(f"\n===== {name} =====")

    display(
        profile.round(2)
    )


===== B_교육생활_GMM_K2 =====


,전용면적,전용면적_IQR,기준연식,층,유초등접근거리_m,역세권거리_m,대규모점포거리_m,응급의료기관거리_m,주요일자리거리_m,단지수,단지비율_%
cluster,,,,,,,,,,,
0,80.43,0.62,22.0,6.0,324.40,473.21,572.66,1102.86,4824.26,7165,94.51
1,76.09,2.22,20.0,8.0,374.82,360.42,98.45,279.75,4452.27,416,5.49



===== C_교통업무_GMM_K2 =====


,전용면적,전용면적_IQR,기준연식,층,유초등접근거리_m,역세권거리_m,대규모점포거리_m,응급의료기관거리_m,주요일자리거리_m,단지수,단지비율_%
cluster,,,,,,,,,,,
0,80.28,0.79,21.0,6.0,316.99,539.86,573.33,1146.89,5979.36,5139,67.79
1,80.20,0.56,22.0,6.5,342.25,298.41,510.35,908.24,1989.90,2442,32.21



===== C_교통업무_KMeans_K2 =====


,전용면적,전용면적_IQR,기준연식,층,유초등접근거리_m,역세권거리_m,대규모점포거리_m,응급의료기관거리_m,주요일자리거리_m,단지수,단지비율_%
cluster,,,,,,,,,,,
0,80.35,0.71,22.0,6.0,316.69,574.05,584.89,1160.96,6077.96,4928,65.0
1,80.09,0.76,22.0,7.0,341.83,287.71,497.20,899.06,2264.18,2653,35.0



===== A_주택구조_KMeans_K3 =====


,전용면적,전용면적_IQR,기준연식,층,유초등접근거리_m,역세권거리_m,대규모점포거리_m,응급의료기관거리_m,주요일자리거리_m,단지수,단지비율_%
cluster,,,,,,,,,,,
0,82.45,24.71,23.0,8.0,312.43,473.45,536.46,1156.93,4542.71,2757,36.37
1,84.15,0.00,22.0,5.0,328.93,486.80,599.03,1064.91,4878.73,3783,49.90
2,36.77,0.48,7.0,9.0,359.19,386.75,430.11,950.51,5397.57,1041,13.73


In [20]:
k3_plus_candidates = (
    cluster_comparison_df[
        cluster_comparison_df["K"] >= 3
    ]
    .sort_values(
        [
            "Silhouette",
            "Davies_Bouldin",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(
    k3_plus_candidates
    .head(20)
    .round(4)
)

,Feature_Set,Algorithm,K,Silhouette,Davies_Bouldin,Calinski_Harabasz
0,C_교통업무,KMeans,3,0.3671,0.9032,5158.0597
1,C_교통업무,KMeans,6,0.3426,0.8578,4947.5662
2,C_교통업무,KMeans,5,0.3370,0.8909,4969.2898
3,C_교통업무,Agglomerative,3,0.3349,1.0024,4273.9584
4,A_주택구조,KMeans,3,0.3320,1.1818,2786.1853
5,A_주택구조,KMeans,5,0.3305,1.1462,2469.6637
6,C_교통업무,GMM,6,0.3279,0.8944,4456.0920
7,C_교통업무,Agglomerative,4,0.3251,1.0692,4097.6721
8,C_교통업무,GMM,3,0.3220,0.9429,4052.5380
9,C_교통업무,KMeans,4,0.3216,0.9581,4833.4308


In [21]:
FINAL_FEATURES = [
    "역세권거리_m",
    "주요일자리거리_m",
]

FINAL_K = 3


X_final, final_scaler = prepare_cluster_data(
    complex_df,
    FINAL_FEATURES,
)


final_cluster_model = KMeans(
    n_clusters=FINAL_K,
    n_init=20,
    random_state=42,
)


final_labels = final_cluster_model.fit_predict(
    X_final
)


complex_df["cluster"] = final_labels


print(
    "Silhouette:",
    f"{silhouette_score(X_final, final_labels):.4f}",
)

print(
    "Davies-Bouldin:",
    f"{davies_bouldin_score(X_final, final_labels):.4f}",
)

print(
    "Calinski-Harabasz:",
    f"{calinski_harabasz_score(X_final, final_labels):.2f}",
)

Silhouette: 0.3725
Davies-Bouldin: 0.9032
Calinski-Harabasz: 5158.06


In [22]:
cluster_size = (
    complex_df["cluster"]
    .value_counts()
    .sort_index()
    .rename("단지수")
    .to_frame()
)

cluster_size["단지비율_%"] = (
    cluster_size["단지수"]
    / len(complex_df)
    * 100
)

display(
    cluster_size.round(2)
)

,단지수,단지비율_%
cluster,,
0,2185,28.82
1,1687,22.25
2,3709,48.92


In [23]:
silhouette_values = silhouette_samples(
    X_final,
    final_labels,
)


cluster_silhouette = []


for cluster in sorted(
    complex_df["cluster"].unique()
):

    mask = (
        complex_df["cluster"].to_numpy()
        == cluster
    )

    cluster_silhouette.append({
        "cluster": cluster,
        "단지수": int(mask.sum()),
        "평균_Silhouette": (
            silhouette_values[mask].mean()
        ),
        "음수비율_%": (
            silhouette_values[mask] < 0
        ).mean() * 100,
    })


cluster_silhouette_df = pd.DataFrame(
    cluster_silhouette
)

display(
    cluster_silhouette_df.round(3)
)

,cluster,단지수,평균_Silhouette,음수비율_%
0,0,2185,0.340,1.281
1,1,1687,0.309,5.572
2,2,3709,0.420,0.000


In [24]:
profile_cols = [
    "역세권거리_m",
    "주요일자리거리_m",

    # 해석용
    "전용면적",
    "기준연식",
    "층",
    "유초등접근거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
]


cluster_profile = (
    complex_df
    .groupby("cluster")[profile_cols]
    .median()
)

cluster_profile["단지수"] = (
    complex_df
    .groupby("cluster")
    .size()
)

cluster_profile["단지비율_%"] = (
    cluster_profile["단지수"]
    / len(complex_df)
    * 100
)


display(
    cluster_profile.round(2)
)

,역세권거리_m,주요일자리거리_m,전용면적,기준연식,층,유초등접근거리_m,대규모점포거리_m,응급의료기관거리_m,단지수,단지비율_%
cluster,,,,,,,,,,
0,266.14,5915.04,76.17,21.0,7.0,322.85,430.95,1047.98,2185,28.82
1,390.62,1522.48,81.84,22.0,6.0,342.64,534.24,888.61,1687,22.25
2,659.26,5766.52,81.95,22.0,6.0,319.17,628.33,1173.56,3709,48.92


In [25]:
cluster_name_map = {
    0: "역세권 생활주거형",
    1: "직주근접 도심형",
    2: "일반 주거형",
}

complex_df["cluster_name"] = (
    complex_df["cluster"]
    .map(cluster_name_map)
)

In [26]:
# ============================================================
# 주택특성 군집 - KMeans K=3
# ============================================================

HOUSING_FEATURES = [
    "전용면적",
    "전용면적_IQR",
    "기준연식",
    "층",
]

HOUSING_K = 3


X_housing, housing_scaler = prepare_cluster_data(
    complex_df,
    HOUSING_FEATURES,
)


housing_model = KMeans(
    n_clusters=HOUSING_K,
    n_init=20,
    random_state=42,
)


housing_labels = housing_model.fit_predict(
    X_housing
)


complex_df["housing_cluster"] = housing_labels


print(
    "Silhouette:",
    f"{silhouette_score(X_housing, housing_labels):.4f}",
)

print(
    "Davies-Bouldin:",
    f"{davies_bouldin_score(X_housing, housing_labels):.4f}",
)

print(
    "Calinski-Harabasz:",
    f"{calinski_harabasz_score(X_housing, housing_labels):.2f}",
)

Silhouette: 0.3320
Davies-Bouldin: 1.1818
Calinski-Harabasz: 2786.19


In [27]:
housing_cluster_size = (
    complex_df["housing_cluster"]
    .value_counts()
    .sort_index()
    .rename("단지수")
    .to_frame()
)

housing_cluster_size["단지비율_%"] = (
    housing_cluster_size["단지수"]
    / len(complex_df)
    * 100
)

display(
    housing_cluster_size.round(2)
)

,단지수,단지비율_%
housing_cluster,,
0,2757,36.37
1,3783,49.90
2,1041,13.73


In [29]:
housing_cluster_size = (
    complex_df["housing_cluster"]
    .value_counts()
    .sort_index()
    .rename("단지수")
    .to_frame()
)

housing_cluster_size["단지비율_%"] = (
    housing_cluster_size["단지수"]
    / len(complex_df)
    * 100
)

display(
    housing_cluster_size.round(2)
)

,단지수,단지비율_%
housing_cluster,,
0,2757,36.37
1,3783,49.90
2,1041,13.73


In [30]:
housing_silhouette_values = silhouette_samples(
    X_housing,
    housing_labels,
)

housing_silhouette_results = []


for cluster in sorted(
    complex_df["housing_cluster"].unique()
):

    mask = (
        complex_df["housing_cluster"].to_numpy()
        == cluster
    )

    housing_silhouette_results.append({
        "cluster": cluster,
        "단지수": int(mask.sum()),
        "단지비율_%": mask.mean() * 100,
        "평균_Silhouette": (
            housing_silhouette_values[mask].mean()
        ),
        "음수비율_%": (
            housing_silhouette_values[mask] < 0
        ).mean() * 100,
    })


housing_silhouette_df = pd.DataFrame(
    housing_silhouette_results
)

display(
    housing_silhouette_df.round(3)
)

,cluster,단지수,단지비율_%,평균_Silhouette,음수비율_%
0,0,2757,36.367,0.277,2.793
1,1,3783,49.901,0.400,0.000
2,2,1041,13.732,0.229,14.697


In [31]:
# ============================================================
# 주택특성 K=5 추가 확인
# ============================================================

housing_model_k5 = KMeans(
    n_clusters=5,
    n_init=20,
    random_state=42,
)

housing_labels_k5 = housing_model_k5.fit_predict(
    X_housing
)

complex_df["housing_cluster_k5"] = housing_labels_k5


sil_values_k5 = silhouette_samples(
    X_housing,
    housing_labels_k5,
)


k5_silhouette_results = []

for cluster in sorted(
    complex_df["housing_cluster_k5"].unique()
):

    mask = (
        complex_df["housing_cluster_k5"].to_numpy()
        == cluster
    )

    k5_silhouette_results.append({
        "cluster": cluster,
        "단지수": int(mask.sum()),
        "단지비율_%": mask.mean() * 100,
        "평균_Silhouette": sil_values_k5[mask].mean(),
        "음수비율_%": (
            sil_values_k5[mask] < 0
        ).mean() * 100,
    })


display(
    pd.DataFrame(
        k5_silhouette_results
    ).round(3)
)

,cluster,단지수,단지비율_%,평균_Silhouette,음수비율_%
0,0,2103,27.740,0.328,0.143
1,1,781,10.302,0.308,7.554
2,2,676,8.917,0.181,14.201
3,3,3427,45.205,0.404,0.175
4,4,594,7.835,0.093,35.185


In [32]:
housing_profile_k5 = (
    complex_df
    .groupby("housing_cluster_k5")[
        [
            "전용면적",
            "전용면적_IQR",
            "기준연식",
            "층",
        ]
    ]
    .median()
)

housing_profile_k5["단지수"] = (
    complex_df
    .groupby("housing_cluster_k5")
    .size()
)

housing_profile_k5["단지비율_%"] = (
    housing_profile_k5["단지수"]
    / len(complex_df)
    * 100
)

display(
    housing_profile_k5.round(2)
)

,전용면적,전용면적_IQR,기준연식,층,단지수,단지비율_%
housing_cluster_k5,,,,,,
0,77.74,24.66,22.0,7.0,2103,27.74
1,27.92,0.38,8.0,8.5,781,10.30
2,69.44,10.54,39.5,6.5,676,8.92
3,84.26,0.00,22.0,5.0,3427,45.21
4,84.94,13.30,18.0,14.0,594,7.84


| 군집          | 목적            | Feature           | 후보                           |
| ----------- | ------------- | ----------------- | ---------------------------- |
| **입지 군집**   | 교통·직주근접 유형    | 역세권거리, 주요일자리거리    | KMeans K=3, Silhouette 0.367 |
| **주택특성 군집** | 신축·구축/주거구조 유형 | 연식, 면적, 면적 IQR, 층 | KMeans K=3, Silhouette 0.332 |
